In [ ]:
import os
import pandas as pd
import numpy as np
from glob import glob
from collections import defaultdict, Counter
from sklearn.model_selection import train_test_split

In [ ]:
# === Set path ===
base_dir = "./ble-accelerometer-indoor-localisation-measurements"
house_dir = os.path.join(base_dir, "house_D")
meta_dir = os.path.join(house_dir, "metadata")
exp_dir = os.path.join(house_dir, "experiments")

# === Load room metadata ===
def load_room_tags(path):
    room_tags = {}
    with open(path, "r") as f:
        for line in f:
            parts = list(map(int, line.strip().split(",")))
            room_id, tag_ids = parts[0], parts[1:]
            for tag in tag_ids:
                room_tags[tag] = room_id
    return room_tags

def load_room_names(path):
    room_names = {}
    with open(path, "r") as f:
        for line in f:
            room_id, room_name = line.strip().split(",")
            room_names[int(room_id)] = room_name
    return room_names

room_tags = load_room_tags(os.path.join(meta_dir, "room_tags.dat"))
room_names = load_room_names(os.path.join(meta_dir, "room_names.dat"))

# === Process all living_* experiments ===
all_data = []

for folder in sorted(glob(os.path.join(exp_dir, "living_*"))):
    rssi_file = os.path.join(folder, "rx_wearable_data.dat")
    tag_file = os.path.join(folder, "tag_annotations.dat")

    if not (os.path.exists(rssi_file) and os.path.exists(tag_file)):
        continue

    rssi_df = pd.read_csv(rssi_file)
    tag_df = pd.read_csv(tag_file)
    rssi_df['timestamp'] = pd.to_datetime(rssi_df['timestamp'])
    tag_df['timestamp'] = pd.to_datetime(tag_df['timestamp'])

    # Bin timestamps to 1-second windows
    rssi_df["time_bin"] = rssi_df["timestamp"].dt.floor("s")
    tag_df["time_bin"] = tag_df["timestamp"].dt.floor("s")

    # For each time bin, take the mode tag
    tag_by_second = tag_df.groupby("time_bin")["tag"].agg(lambda x: x.mode().iloc[0]).reset_index()

    # Merge to assign tag to RSSI values
    merged = pd.merge(rssi_df, tag_by_second, on="time_bin", how="inner")

    # Map tag → room_id → room_name
    merged["room_id"] = merged["tag"].map(room_tags)
    merged["room_name"] = merged["room_id"].map(room_names)

    all_data.append(merged)

# === Combine and pivot ===
full_df = pd.concat(all_data, ignore_index=True)

# Pivot RSSI values across APs, 1 row per second
pivot = full_df.pivot_table(index="time_bin", columns="ap_id", values="rssi", aggfunc="mean")
pivot.columns = [f"rssi_ap{int(c)}" for c in pivot.columns]
pivot.fillna(-120, inplace=True)

# Add room name label
room_labels = full_df.groupby("time_bin")["room_name"].agg(lambda x: x.mode().iloc[0])
pivot["room_name"] = room_labels

# === Save to CSV (optional) ===
pivot.reset_index().to_csv("house_D_rssi_labeled.csv", index=False)

# Preview first few rows
pivot.reset_index().head()

✅ Saved: house_D_rssi_labeled.csv


,time_bin,rssi_ap1,rssi_ap2,rssi_ap3,rssi_ap4,rssi_ap5,rssi_ap6,rssi_ap7,rssi_ap8,rssi_ap9,rssi_ap10,rssi_ap11,room_name
0,2017-04-10 13:33:14+00:00,-85.00,-79.000000,-84.500000,-88.00,-82.000000,-87.666667,-71.750000,-69.00,-75.666667,-78.000000,-53.00,living_area_A
1,2017-04-10 13:33:15+00:00,-88.80,-84.333333,-74.000000,-84.75,-82.250000,-83.000000,-69.000000,-68.00,-70.333333,-85.000000,-59.25,living_area_A
2,2017-04-10 13:33:16+00:00,-79.00,-84.000000,-82.333333,-85.50,-88.500000,-79.500000,-70.000000,-69.00,-70.000000,-82.333333,-52.25,living_area_A
3,2017-04-10 13:33:17+00:00,-86.75,-85.500000,-76.000000,-82.20,-85.750000,-82.000000,-72.666667,-66.75,-70.750000,-82.333333,-61.00,living_area_A
4,2017-04-10 13:33:18+00:00,-92.00,-76.666667,-78.000000,-82.50,-85.666667,-83.333333,-73.666667,-66.20,-68.500000,-86.500000,-53.60,living_area_A


In [ ]:
def load_full_dataset(path):
    pivot = pd.read_csv(path)
    if 'room_name' in pivot.columns:
        pivot = pivot.rename(columns={'room_name': 'label'})
    return pivot

def split_by_date(pivot):
    # Extract just the date part
    pivot['date'] = pd.to_datetime(pivot['time_bin']).dt.date

    # Count number of samples per date
    date_counts = pivot['date'].value_counts()

    # Select training date (more samples) and validation date
    train_date = date_counts.idxmax()
    val_date = date_counts.idxmin()

    print(f"Training Date: {train_date} ({date_counts[train_date]} samples)")
    print(f"Validation Date: {val_date} ({date_counts[val_date]} samples)")

    pivot_train = pivot[pivot['date'] == train_date].reset_index(drop=True)
    pivot_val = pivot[pivot['date'] == val_date].reset_index(drop=True)

    return pivot_train, pivot_val

def window_and_scale_data(pivot, window_size=5, hop_size=1, min_val=None, max_val=None):
    features = pivot.drop(columns=["label"]).values
    labels = pivot["label"].values

    if min_val is None or max_val is None:
        # MinMax scaling from scratch (training)
        min_val = features.min(axis=0)
        max_val = features.max(axis=0)

    # Apply given min_val and max_val
    features_scaled = (features - min_val) / (max_val - min_val + 1e-8)

    # Windowing
    X_win, y_win = [], []
    for i in range(0, len(features_scaled) - window_size + 1, hop_size):
        window = features_scaled[i:i+window_size]
        window_labels = labels[i:i+window_size]
        if np.all(window_labels == window_labels[0]):
            X_win.append(window)
            y_win.append(window_labels[0])

    X_win = np.stack(X_win)
    y_win = np.array(y_win)

    return X_win, y_win, min_val, max_val

def save_windowed_dataset(X, y, save_path):
    np.savez_compressed(save_path, X=X, y=y)

import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split

def create_experiment_splits_by_date(full_csv_path, output_dir="house_D_windowed_splits"):
    os.makedirs(output_dir, exist_ok=True)

    house_d_label_to_id = {
        "living_area_A": 0,
        "kitchen": 1,
        "living_area_B": 2,
        "hallway_upper": 3,
        "bedroom_1": 4,
        "bedroom_2": 5,
        "hallway_lower": 6,
        "stairs": 7,
        "bathroom_toilet": 8,
        "outside": 9
    }

    pivot = load_full_dataset(full_csv_path)
    pivot["label"] = pivot["label"].map(house_d_label_to_id)

    # Define number of classes
    NUM_CLASSES = len(house_d_label_to_id)

    pivot_train, pivot_val = split_by_date(pivot)

    pivot_train = pivot_train.drop(columns=["time_bin", "date"])
    pivot_val = pivot_val.drop(columns=["time_bin", "date"])

    # Window and scale
    X_train_win, y_train_win, min_val, max_val = window_and_scale_data(pivot_train)
    X_val_win, y_val_win, _, _ = window_and_scale_data(pivot_val, min_val=min_val, max_val=max_val)

    # Save evaluation set (from other date)
    save_windowed_dataset(X_val_win, y_val_win, os.path.join(output_dir, "houseD_full_validation_set.npz"))

    labeled_ratios = [0.20, 0.10, 0.05, 0.01]
    rare_class_ids = [3, 6, 7, 9]  # hallway_upper, hallway_lower, stairs, outside

    for exp_num, ratio in enumerate(labeled_ratios, start=1):

        # --- Separate rare and normal samples ---
        rare_mask = np.isin(y_train_win, rare_class_ids)
        normal_mask = ~rare_mask

        X_rare = X_train_win[rare_mask]
        y_rare = y_train_win[rare_mask]

        X_normal = X_train_win[normal_mask]
        y_normal = y_train_win[normal_mask]

        # --- Split rare classes manually into thirds ---
        X_labeled_rare = []
        y_labeled_rare = []
        X_unlabeled_rare = []
        y_unlabeled_rare = []
        X_val_rare = []
        y_val_rare = []

        for class_id in rare_class_ids:
            class_mask = (y_rare == class_id)
            class_X = X_rare[class_mask]
            class_y = y_rare[class_mask]

            n = len(class_X)
            idx = np.random.permutation(n)

            n_third = n // 3
            remainder = n % 3

            idx_labeled = idx[:n_third]
            idx_unlabeled = idx[n_third:2*n_third]
            idx_val = idx[2*n_third:]

            # Handle leftovers: prioritize labeled -> unlabeled
            if remainder == 1:
                idx_labeled = np.append(idx_labeled, idx_val[0])
                idx_val = idx_val[1:]
            elif remainder == 2:
                idx_labeled = np.append(idx_labeled, idx_val[0])
                idx_unlabeled = np.append(idx_unlabeled, idx_val[1])
                idx_val = idx_val[2:]

            X_labeled_rare.append(class_X[idx_labeled])
            y_labeled_rare.append(class_y[idx_labeled])

            X_unlabeled_rare.append(class_X[idx_unlabeled])
            y_unlabeled_rare.append(class_y[idx_unlabeled])

            X_val_rare.append(class_X[idx_val])
            y_val_rare.append(class_y[idx_val])

        X_labeled_rare = np.concatenate(X_labeled_rare) if X_labeled_rare else np.empty((0, X_train_win.shape[1], X_train_win.shape[2]))
        y_labeled_rare = np.concatenate(y_labeled_rare) if y_labeled_rare else np.empty((0,), dtype=int)

        X_unlabeled_rare = np.concatenate(X_unlabeled_rare) if X_unlabeled_rare else np.empty((0, X_train_win.shape[1], X_train_win.shape[2]))
        y_unlabeled_rare = np.concatenate(y_unlabeled_rare) if y_unlabeled_rare else np.empty((0,), dtype=int)

        X_val_rare = np.concatenate(X_val_rare) if X_val_rare else np.empty((0, X_train_win.shape[1], X_train_win.shape[2]))
        y_val_rare = np.concatenate(y_val_rare) if y_val_rare else np.empty((0,), dtype=int)

        # --- Normal samples stratified split ---
        X_labeled_normal, X_unlabeled_normal, y_labeled_normal, y_unlabeled_normal = train_test_split(
            X_normal, y_normal, train_size=ratio, random_state=42, stratify=y_normal
        )

        # --- Merge labeled and unlabeled ---
        X_labeled_final = np.concatenate([X_labeled_normal, X_labeled_rare])
        y_labeled_final = np.concatenate([y_labeled_normal, y_labeled_rare])

        X_unlabeled_final = np.concatenate([X_unlabeled_normal, X_unlabeled_rare])
        y_unlabeled_final = np.concatenate([y_unlabeled_normal, y_unlabeled_rare])

        # --- Merge validation set (original val + rare samples) ---
        X_val_final = np.concatenate([X_val_win, X_val_rare])
        y_val_final = np.concatenate([y_val_win, y_val_rare])

        # --- Save splits ---
        save_windowed_dataset(X_labeled_final, y_labeled_final, os.path.join(output_dir, f"houseD_labeled_experiment{exp_num}.npz"))
        save_windowed_dataset(X_unlabeled_final, y_unlabeled_final, os.path.join(output_dir, f"houseD_unlabeled_experiment{exp_num}_train.npz"))
        if exp_num == 1:  # Save the validation set once
            save_windowed_dataset(X_val_final, y_val_final, os.path.join(output_dir, "houseD_full_validation_set.npz"))

        # --- Print counts ---
        labeled_counts = np.bincount(y_labeled_final, minlength=NUM_CLASSES)
        print("\nNumber of datapoints per class in Labeled Train Set:")
        for idx, count in enumerate(labeled_counts):
            print(f"Class {idx}: {count}")

        unlabeled_counts = np.bincount(y_unlabeled_final, minlength=NUM_CLASSES)
        print("\nNumber of datapoints per class in Unlabeled Train Set:")
        for idx, count in enumerate(unlabeled_counts):
            print(f"Class {idx}: {count}")

        val_counts = np.bincount(y_val_final, minlength=NUM_CLASSES)
        print("\nNumber of datapoints per class in Validation Set:")
        for idx, count in enumerate(val_counts):
            print(f"Class {idx}: {count}")


In [31]:
create_experiment_splits_by_date("house_D_rssi_labeled.csv")

Training Date: 2017-04-10 (4683 samples)
Validation Date: 2017-04-13 (2066 samples)
✅ Saved: house_D_windowed_splits/houseD_full_validation_set.npz

🔵 Creating Experiment 1 with 20% labeled windows...
✅ Saved: house_D_windowed_splits/houseD_labeled_experiment1.npz
✅ Saved: house_D_windowed_splits/houseD_unlabeled_experiment1_train.npz
✅ Saved: house_D_windowed_splits/houseD_full_validation_set.npz

Number of datapoints per class in Labeled Train Set:
Class 0: 163
Class 1: 296
Class 2: 31
Class 3: 18
Class 4: 113
Class 5: 117
Class 6: 13
Class 7: 5
Class 8: 23
Class 9: 5

Number of datapoints per class in Unlabeled Train Set:
Class 0: 653
Class 1: 1184
Class 2: 122
Class 3: 17
Class 4: 455
Class 5: 467
Class 6: 13
Class 7: 5
Class 8: 92
Class 9: 4

Number of datapoints per class in Validation Set:
Class 0: 938
Class 1: 362
Class 2: 232
Class 3: 26
Class 4: 106
Class 5: 33
Class 6: 13
Class 7: 15
Class 8: 152
Class 9: 9

🔵 Creating Experiment 2 with 10% labeled windows...
✅ Saved: house_